In [161]:
%load_ext autoreload
%autoreload 2

from smac.env import StarCraft2Env
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [162]:
def lowest_health_attack(env, a):
    avail = np.nonzero(env.get_avail_agent_actions(a))[0]
    offset = env.n_actions_no_attack          # = 6
    attacks = avail[avail >= offset]
    if len(attacks) == 0:
        return None                            # no enemy in range -> use your movement logic

    def hp(action):
        e = env.enemies[action - offset]
        return e.health + e.shield             # shield is 0 for non-Protoss units

    return min(attacks, key=hp)

def scripted_action(env, a, focus_fire):
    avail = np.nonzero(env.get_avail_agent_actions(a))[0]
    attacks = avail[avail >= 6]
    if len(attacks):
        if focus_fire:
            return lowest_health_attack(env,a)
        else:    
            return np.random.choice(attacks)   # or pick the lowest-health target for "focus fire"
    unit = env.agents[a]
    if unit.health == 0:
        return 0                            # no-op for dead units
    alive = [e for e in env.enemies.values() if e.health > 0]
    cx = np.mean([e.pos.x for e in alive]); cy = np.mean([e.pos.y for e in alive]) # You move towards the average position of all enemies
    dx, dy = cx - unit.pos.x, cy - unit.pos.y
    move = (4 if dx > 0 else 5) if abs(dx) > abs(dy) else (2 if dy > 0 else 3)  # E/W or N/S
    return move if move in avail else avail[0]



def log_state(rows, ep, t, env, actions=None):
    for team, units in (("ally", env.agents), ("enemy", env.enemies)):
        for uid, u in units.items():
            a = actions[uid] if (team == "ally" and actions is not None) else None
            rows.append(dict(
                episode_id=ep, t=t, team=team, unit_id=uid,
                health=u.health, shield=u.shield, alive=u.health > 0,
                x=u.pos.x, y=u.pos.y,
                action=a, target_id=(a - 6 if a is not None and a >= 6 else None),
            ))

def simulate_battle(env, ep, rng, random_behav=False, focus_fire=False):
    rows = []
    env.reset()
    done, t, info = False, 0, {}
    while not done:
        if random_behav:
            actions = [rng.choice(np.nonzero(env.get_avail_agent_actions(i))[0])
                       for i in range(env.n_agents)]
        else:
            actions = [scripted_action(env, i, focus_fire) for i in range(env.n_agents)]
        log_state(rows, ep, t, env, actions)      # state *before* the action
        _, done, info = env.step(actions)
        t += 1
    log_state(rows, ep, t, env)                    # final state
    meta = dict(episode_id=ep, random_behav=random_behav, focus_fire=focus_fire,
                battle_won=info.get("battle_won", False),
                timeout=info.get("episode_limit", False), length=t)
    return rows, meta

In [163]:
import pandas as pd

N_SIMULATIONS = 100


env = StarCraft2Env(map_name="8m", step_mul=2, seed=0)
rng = np.random.default_rng(0)
all_rows, all_meta = [], []
for ep in range(N_SIMULATIONS):
    rows, meta = simulate_battle(env, ep, rng, focus_fire=True)
    all_rows += rows; all_meta.append(meta)
env.close()

units = pd.DataFrame(all_rows)
episodes = pd.DataFrame(all_meta)


units.to_csv(f"data/units_{N_SIMULATIONS}_episodes.csv", index=False)
episodes.to_csv(f'data/meta_{N_SIMULATIONS}_episodes.csv', index=False)

In [164]:
episodes.head()

,episode_id,random_behav,focus_fire,battle_won,timeout,length
0,0,False,True,True,False,97
1,1,False,True,True,False,99
2,2,False,True,True,False,90
3,3,False,True,True,False,98
4,4,False,True,True,False,91


In [165]:
agg = (units.groupby(["episode_id", "t", "team"])
            .agg(hp=("health", "sum"), alive=("alive", "sum"))
            .unstack("team"))
agg.head()

hp        alive      
team           ally  enemy  ally enemy
episode_id t                          
0          0  360.0  360.0     8     8
           1  360.0  360.0     8     8
           2  360.0  360.0     8     8
           3  360.0  360.0     8     8
           4  360.0  360.0     8     8